 TransE with hierarchy-aware entity embeddings

This notebook does two things:

1. trains a normal TransE model on the taxonomy graph
2. fuses each entity embedding with the average vector of the hierarchy relations attached to that entity

This preserves the TransE relation modeling, but gives each entity a representation that also contains its hypernym/hyponym topology.

The fusion is:

e'_entity = e_entity + alpha * mean(relation_vectors_for_entity)

where the relation vectors come from the TransE relation embeddings for hypernym/hyponym edges.


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rdflib import Graph, Literal, URIRef
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory

In [ ]:
# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
RDF_PATH = Path("wordnet_nouns.ttl")
CSV_PATH = Path("animals_oewn.csv")

TAXONOMY_PREDICATES = {
    "https://globalwordnet.github.io/schemas/wn#hypernym",
    "https://globalwordnet.github.io/schemas/wn#hyponym",
    "https://en-word.net/ontology/hypernym",
    "https://en-word.net/ontology/hyponym",
}

EMBEDDING_DIMENSION = 100
EPOCHS = 100
RANDOM_SEED = 42
ALPHA = 0.5  # strength of hierarchy context
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


In [ ]:
# ------------------------------------------------------------
# Load RDF and keep only taxonomy relations
# ------------------------------------------------------------
g = Graph()
g.parse(str(RDF_PATH), format="turtle")

triples = []

for s, p, o in g:
    pred = str(p)
    if pred not in TAXONOMY_PREDICATES:
        continue

    # Keep only URI-based members in the graph
    if isinstance(o, Literal):
        continue

    triples.append((str(s), str(p), str(o)))

print(f"Triples loaded from RDF: {len(g)}")
print(f"Taxonomy triples retained: {len(triples)}")

# quick sanity check
from collections import Counter
pred_counts = Counter(t[1] for t in triples)
print("Predicate counts:")
for k, v in pred_counts.most_common():
    print(f"  {k}: {v}")


In [ ]:
# ------------------------------------------------------------
# Build the PyKEEN triples factory
# ------------------------------------------------------------
# Each element is (head, relation, tail)
triples_array = np.array(triples, dtype=object)

triples_factory = TriplesFactory.from_labeled_triples(
    triples=triples_array,
    create_inverse_triples=False,
)

print(f"Entities: {len(triples_factory.entity_to_id)}")
print(f"Relations: {len(triples_factory.relation_to_id)}")
print(triples_factory.relation_to_id)


In [ ]:
# ------------------------------------------------------------
# Train a standard TransE model
# ------------------------------------------------------------
result = pipeline(
    training=triples_factory,
    testing=triples_factory,
    validation=triples_factory,
    model="TransE",
    model_kwargs={
        "embedding_dim": EMBEDDING_DIMENSION,
    },
    training_kwargs={
        "epochs": EPOCHS,
        "batch_size": 256,
        "label_smoothing": 0.0,
    },
    negative_sampler="basic",
    optimizer="Adam",
    random_seed=RANDOM_SEED,
    device=DEVICE,
)

model = result.model
print("Training finished.")
print("Loss:", result.training.losses[-1] if len(result.training.losses) > 0 else "n/a")


In [ ]:
# ------------------------------------------------------------
# Extract entity and relation embeddings from the trained model
# ------------------------------------------------------------
entity_embedding = model.entity_representations[0].weight.detach().cpu().numpy()
relation_embedding = model.relation_representations[0].weight.detach().cpu().numpy()

entity_id_to_index = {entity: idx for entity, idx in triples_factory.entity_to_id.items()}
relation_id_to_index = {rel: idx for rel, idx in triples_factory.relation_to_id.items()}

print("Entity embedding shape:", entity_embedding.shape)
print("Relation embedding shape:", relation_embedding.shape)


In [ ]:
# ------------------------------------------------------------
# Build hierarchy-aware context for each entity
# ------------------------------------------------------------
# For each entity, collect the relation vectors connected to it through hypernym/hyponym edges.
# Then average them.

entity_hierarchy_context = {e: [] for e in triples_factory.entity_to_id.keys()}

for h, r, t in triples:
    rel_uri = r

    if rel_uri not in TAXONOMY_PREDICATES:
        continue

    h_id = triples_factory.entity_to_id.get(h)
    t_id = triples_factory.entity_to_id.get(t)
    r_id = triples_factory.relation_to_id.get(rel_uri)

    if h_id is None or t_id is None or r_id is None:
        continue

    entity_hierarchy_context[h].append(r_id)
    entity_hierarchy_context[t].append(r_id)

# compute average relation-context vector per entity
entity_context_vectors = {}

for entity, rel_ids in entity_hierarchy_context.items():
    if not rel_ids:
        entity_context_vectors[entity] = np.zeros(EMBEDDING_DIMENSION, dtype=np.float32)
    else:
        rel_vecs = relation_embedding[np.array(rel_ids, dtype=int)]
        entity_context_vectors[entity] = rel_vecs.mean(axis=0)

print("Example entity contexts:")
for entity in list(entity_hierarchy_context.keys())[:3]:
    print(entity, entity_context_vectors[entity][:5])


In [ ]:
# ------------------------------------------------------------
# Fuse entity embedding with hierarchy context
# ------------------------------------------------------------
fused_entity_embedding = np.zeros_like(entity_embedding, dtype=np.float32)

for entity, idx in entity_id_to_index.items():
    base_vec = entity_embedding[idx]
    context_vec = entity_context_vectors.get(entity, np.zeros(EMBEDDING_DIMENSION, dtype=np.float32))
    fused_entity_embedding[idx] = base_vec + ALPHA * context_vec

print("Fused entity embedding shape:", fused_entity_embedding.shape)
print("Sample fused vector norm:", np.linalg.norm(fused_entity_embedding[0]))


In [ ]:
# ------------------------------------------------------------
# Save fused embeddings
# ------------------------------------------------------------
entity_names = list(triples_factory.entity_to_id.keys())

fused_df = pd.DataFrame(fused_entity_embedding, index=entity_names)
# if you want columns names:
fused_df.columns = [f"dim_{i}" for i in range(fused_entity_embedding.shape[1])]

# add back the entity column
fused_df.insert(0, "entity", entity_names)

fused_df.head()


In [ ]:
# ------------------------------------------------------------
# Optional: compute cosine similarity on the fused embeddings
# ------------------------------------------------------------
# This is useful if you want to compare synsets from animals_oewn.csv

df = pd.read_csv(CSV_PATH)

def normalize_synset(value):
    if pd.isna(value):
        return None
    synset = str(value).strip()
    if synset.startswith("http://") or synset.startswith("https://"):
        synset = synset.rstrip("/").split("/")[-1]
    if not synset.startswith("oewn-"):
        return None
    return f"https://en-word.net/id/{synset}"

def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

results = []

for _, row in df.iterrows():
    noun_synset = normalize_synset(row.get("noun_synset"))
    hyper_synset = normalize_synset(row.get("hypernym_synset"))
    hypo_synset = normalize_synset(row.get("hyponym_synset"))

    if noun_synset is None or hyper_synset is None:
        continue

    if noun_synset not in triples_factory.entity_to_id or hyper_synset not in triples_factory.entity_to_id:
        continue

    noun_idx = triples_factory.entity_to_id[noun_synset]
    hyper_idx = triples_factory.entity_to_id[hyper_synset]

    noun_vec = fused_entity_embedding[noun_idx]
    hyper_vec = fused_entity_embedding[hyper_idx]

    sim_hyper = cosine_similarity(noun_vec, hyper_vec)

    sim_hypo = None
    if hypo_synset is not None and hypo_synset in triples_factory.entity_to_id:
        hypo_idx = triples_factory.entity_to_id[hypo_synset]
        hypo_vec = fused_entity_embedding[hypo_idx]
        sim_hypo = cosine_similarity(noun_vec, hypo_vec)

    results.append({
        "noun": row.get("noun"),
        "noun_synset": noun_synset,
        "hypernym": row.get("hypernym"),
        "hypernym_synset": hyper_synset,
        "noun_hypernym_similarity": sim_hyper,
        "hyponym": row.get("hyponym"),
        "hyponym_synset": hypo_synset,
        "noun_hyponym_similarity": sim_hypo,
    })

sim_df = pd.DataFrame(results)
print(sim_df.head())
print(f"Rows computed: {len(sim_df)}")


# Interpretation

This setup keeps the standard TransE structure:

- entity embeddings are learned
- relation embeddings are learned
- the scoring function remains the usual TransE scoring

But then it enriches every entity with its hierarchy context:

- for each entity, collect the hypernym and hyponym relation vectors
- average them
- add them to the original entity vector

This makes the final entity representation aware of its place in the taxonomy while still being rooted in the TransE embedding space.
